In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.nn.functional import scaled_dot_product_attention
import math
import time
import os
from torch.cuda.amp import GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
from transformers import GPT2Tokenizer
from torch.utils.data import DataLoader

class LlamaRMSNorm(nn.Module):
    """
        LlamaRMSNorm is equivalent to T5LayerNorm.
        x / sqrt(mean(x^2, axis=-1) + eps) * weight
    """
    
    def __init__(self, d_model, eps=1e-5):
        """
        Args:
            d_model (int)  : Dimensionality of the input tensor.
            eps     (float): A small constant added to avoid division by zero.
        """
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x): # return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * x / norm
    
class LlamaRotaryEmbedding(nn.Module):
    """
    # https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py#L240
    Rotary Positional Embedding (RoPE) for transformers Implemntation derived from https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py
    """
    def __init__(self, dim: int, theta: float = 10000.0):
        super().__init__()
        self.dim = dim
        self.theta = theta

    def forward(self, x: torch.Tensor, seq_len: int) -> torch.Tensor:
       
        B, T, H, H_D = x.shape

        # Generate position indices
        position = torch.arange(T, dtype=torch.bfloat16, device=x.device).unsqueeze(-1)

        # Generate frequencies
        freqs = torch.exp(
            torch.arange(0, H_D, 2, dtype=torch.bfloat16, device=x.device) * 
            -(torch.log(torch.tensor(self.theta)) / H_D)
                                                                
        )

        # Compute sinusoids
        sinusoid = position * freqs
        sin = torch.sin(sinusoid)
        cos = torch.cos(sinusoid)

        # Reshape sin and cos to match the input tensor's shape
        sin = sin.unsqueeze(0).unsqueeze(2)  # Shape: (1, T, 1, D // 2)
        cos = cos.unsqueeze(0).unsqueeze(2)  # Shape: (1, T, 1, D // 2)

        # Apply rotary embeddings
        x_rotated = x.clone()
        x_rotated[..., 0::2] = x[..., 0::2] * cos - x[..., 1::2] * sin
        x_rotated[..., 1::2] = x[..., 1::2] * cos + x[..., 0::2] * sin

        return x_rotated
        
class LlamaSdpaAttention(nn.Module):
    """
    Implements multi-head self-attention with rotary embeddings and scaled dot-product attention.
    """
    
    def __init__(self, hidden_size, num_heads):
        """
        Args:
            hidden_size (int): Total dimensionality of the input embeddings.
            num_heads   (int): Number of attention heads.
        """
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.num_key_value_heads = 3

        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, self.head_dim * self.num_key_value_heads, bias=False)
        self.v_proj = nn.Linear(hidden_size, self.head_dim * self.num_key_value_heads, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.rotary_emb = LlamaRotaryEmbedding(self.head_dim)

    def forward(self, x, attention_mask=None):
        """
        Args:
            x (torch.Tensor): Input tensor of shape [batch_size, seq_len, hidden_size].
            attention_mask (torch.Tensor, optional): Mask to control which positions are attended to.
                                                     Shape [batch_size, seq_len].

        Returns:
            torch.Tensor: The output tensor after applying multi-head attention, of shape [batch_size, seq_len, hidden_size].
        """
        batch_size, seq_len, _ = x.shape
        B, T, C = x.size()

        # Project queries, keys and values
        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_key_value_heads, self.head_dim)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_key_value_heads, self.head_dim)

        # Reshape for attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        q = self.rotary_emb(q, T)
        k = self.rotary_emb(k, T)

        # Repeat k/v heads if num_key_value_heads < num_attention_heads
        if self.num_key_value_heads != self.num_heads:
            k = k.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=1) # B,kv_head,T,D -> B,H,T,D
            v = v.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=1) # B,kv_head,T,D -> B,H,T,D

        # Scaled dot product attention
        attn_output = scaled_dot_product_attention(
            q, k, v,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=True
        )

        # Reshape and project output
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        return self.o_proj(attn_output)

class LlamaMLP(nn.Module):
    """
    Implements the MLP (Multilayer Perceptron) block used in LLaMA transformer layers.
    The MLP block applies a feed-forward network with non-linearity and projections.
    """
    
    def __init__(self, hidden_size, intermediate_size):
        """
        Initializes the MLP block with three linear layers and an activation function.

        Args:
            hidden_size (int): Dimensionality of the input and output embeddings.
            intermediate_size (int): Dimensionality of the intermediate feed-forward layer.
        """
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x):
        """
        Args:
            x (torch.Tensor): Input tensor of shape [batch_size, seq_len, hidden_size].

        Returns:
            torch.Tensor: Output tensor of shape [batch_size, seq_len, hidden_size].
        """
        return self.down_proj(self.act_fn(self.gate_proj(x) + self.up_proj(x)))
    
class LlamaDecoderLayer(nn.Module):
    """
    Implements a single transformer decoder layer used in the LLaMA model. Each layer
    consists of a self-attention mechanism, followed by a feed-forward network (MLP),
    with residual connections and normalization applied at steps.
    """
    
    def __init__(self, hidden_size, num_heads, intermediate_size):
        """
        Args:
            hidden_size (int): Dimensionality of input and output embeddings.
            num_heads (int): Number of attention heads for multi-head self-attention.
            intermediate_size (int): Dimensionality of the feed-forward network.
        """
        super().__init__()
        self.attn = LlamaSdpaAttention(hidden_size, num_heads)
        self.input_layernorm = LlamaRMSNorm(hidden_size)
        self.post_attention_layernorm = LlamaRMSNorm(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)
    
    def forward(self, x, attention_mask=None):
        """
        Forward pass for a single decoder layer.

        Args:
            x (torch.Tensor): Input tensor of shape [batch_size, seq_len, hidden_size].
            attention_mask (torch.Tensor, optional): Mask for controlling which positions to attend to.
                                                     Shape [batch_size, seq_len].

        Returns:
            torch.Tensor: Output tensor of shape [batch_size, seq_len, hidden_size].
        """
        # Self-attention
        residual = x
        x = self.attn(self.input_layernorm(x), attention_mask)
        x = x + residual

        # Feedforward
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = x + residual

        return x
    
class LlamaModel(nn.Module):
    """
    Implements the LLaMA transformer model, consisting of an embedding layer,
    a stack of decoder layers, and an output normalization layer.
    """
    
    def __init__(self, vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size):
        """
        Args:
            vocab_size (int): Size of the input vocabulary.
            hidden_size (int): Dimensionality of the input embeddings and hidden states.
            num_hidden_layers (int): Number of stacked decoder layers in the model.
            num_heads (int): Number of attention heads in each decoder layer.
            intermediate_size (int): Dimensionality of the feedforward layers in the MLPs.
        """
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            LlamaDecoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_hidden_layers)
        ])
        self.norm = LlamaRMSNorm(hidden_size)
    
    def forward(self, input_ids, attention_mask=None):
        """
        Args:
            input_ids (torch.Tensor): Input tensor of token IDs of shape [batch_size, seq_len].
            attention_mask (torch.Tensor, optional): Attention mask to control which tokens are attended to.
                                                     Shape [batch_size, seq_len].

        Returns:
            torch.Tensor: The final hidden states of shape [batch_size, seq_len, hidden_size].
        """
        x = self.embed_tokens(input_ids)

        for layer in self.layers:
            x = layer(x, attention_mask)

        return self.norm(x)
    
class LlamaForCausalLM(nn.Module):
    """
    LLaMA model for causal language modeling. It integrates the
    transformer model (`LlamaModel`) with an additional output head for predicting 
    the next token in a sequence (causal language modeling task). The model shares weights 
    between the token embeddings and the output logits layer (lm_head).
    """
    
    def __init__(self, vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size, max_seq_len):
        """
        Args:
            vocab_size (int): The size of the input vocabulary.
            hidden_size (int): Dimensionality of the input embeddings and hidden states.
            num_hidden_layers (int): Number of stacked decoder layers.
            num_heads (int): Number of attention heads for each decoder layer.
            intermediate_size (int): Size of the intermediate layer in the MLPs.
            max_seq_len (int): Maximum sequence length.
        """
        super().__init__()
        self.model = LlamaModel(vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.model.embed_tokens.weight # Weight Sharing

    def forward(self, input_ids, attention_mask=None):
        """
        Args:
            input_ids (torch.Tensor): Input tensor of token IDs with shape [batch_size, seq_len].
            attention_mask (torch.Tensor, optional): Attention mask to control which tokens are attended to.
                                                     Shape [batch_size, seq_len].

        Returns:
            torch.Tensor: Logits for each token in the vocabulary with shape [batch_size, seq_len, vocab_size].
        """
        x = self.model(input_ids, attention_mask)
        logits = self.lm_head(x)
        return logits
    
    def resize_token_embeddings(self, new_num_tokens):
        """
        Args:
            new_num_tokens (int): The new number of tokens in the vocabulary.

        Returns:
            torch.nn.Embedding: The new token embedding layer with the updated vocabulary size.
        """
        # Resize the token embeddings
        old_embeddings = self.model.embed_tokens
        new_emmbeddings = nn.Embedding(new_num_tokens, old_embeddings.embedding_dim)

        # Copy the exisiting embeddings
        num_tokens_to_copy = min(old_embeddings.num_embeddings, new_num_tokens)
        new_emmbeddings.weight.data[:num_tokens_to_copy, :] = old_embeddings.weight.data[:num_tokens_to_copy, :]

        # Replace the old embeddings
        self.model.embed_tokens = new_emmbeddings

        # Update the lm_head to match the new embedding size
        old_lm_head = self.lm_head
        self.lm_head = nn.Linear(old_lm_head.in_features, new_num_tokens, bias=False)

        # Copy the exisiting weights
        num_tokens_to_copy = min(old_lm_head.out_features, new_num_tokens)
        self.lm_head.weight.data[:num_tokens_to_copy, :] = old_lm_head.weight.data[:num_tokens_to_copy, :]

        # Maintain weight sharing
        self.lm_head.weight = self.model.embed_tokens.weight

        return self.model.embed_tokens

In [3]:
import torch
from transformers import GPT2Tokenizer
import os

def load_checkpoint(model, chkpt_path, device):
    """
    Load only the model's state_dict from a checkpoint file.
    """
    if os.path.exists(chkpt_path):
        checkpoint = torch.load(chkpt_path, map_location=device)
        model.load_state_dict(checkpoint)
        print(f"Checkpoint loaded: {chkpt_path}")
        model = model.to(device)  # Ensure model is on the correct device
        model.eval()  # Set to evaluation mode
    else:
        raise FileNotFoundError(f"Checkpoint file not found: {chkpt_path}")

    return model

def generate_text(model, tokenizer, prompt, max_length=100, temperature=0.7, device='cuda'):
    """Generate text based on a prompt"""
    # Ensure model is on the correct device
    model = model.to(device)
    
    # Encode the prompt
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
    # Move input tensors to the same device as the model
    input_ids = inputs["input_ids"].to(device)
    
    with torch.no_grad():
        # Get the model outputs
        outputs = model(input_ids)
        
        # Get the next token predictions
        next_token_logits = outputs[..., -1, :]
        
        # Apply temperature
        next_token_logits = next_token_logits / temperature
        
        # Sample from the distribution
        probs = torch.nn.functional.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        generated_ids = torch.cat([input_ids, next_token], dim=-1)
        
        # Continue generating until max_length or EOS
        for _ in range(max_length - len(generated_ids[0])):
            outputs = model(generated_ids)
            next_token_logits = outputs[..., -1, :] / temperature
            probs = torch.nn.functional.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated_ids = torch.cat([generated_ids, next_token], dim=-1)
            
            if next_token.item() == tokenizer.eos_token_id:
                break
    
    # Decode the generated text
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return generated_text

def main():
    # Configuration
    checkpoint_path = "checkpoints/model_step_5000.pt"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Initialize tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained("HuggingFaceTB/cosmo2-tokenizer")
    tokenizer.pad_token = tokenizer.eos_token
    model_vocab_size = len(tokenizer)

    # Model configuration
    vocab_size = model_vocab_size
    hidden_size = 576
    num_hidden_layers = 30
    num_heads = 9
    intermediate_size = 1536
    max_seq_len = 2048

    # Initialize model and move to device
    model = LlamaForCausalLM(vocab_size, hidden_size, num_hidden_layers, num_heads, 
                            intermediate_size, max_seq_len)
    model = model.to(device)

    # Load model weights
    print("Loading model from checkpoint...")
    model = load_checkpoint(model, checkpoint_path, device)

    # Print model size
    model_size = sum(p.numel() for p in model.parameters())
    print(f"Model Size: {model_size / 1e6:.2f}M parameters")
    
    # Example prompts for testing
    test_prompts = [
        "Once upon a time",
        "The future of artificial intelligence",
        "In the distant galaxy"
    ]
    
    # Generate text for each prompt
    print("\nGenerating text...\n")
    for prompt in test_prompts:
        print(f"Prompt: {prompt}")
        generated_text = generate_text(
            model,
            tokenizer,
            prompt,
            max_length=100,
            temperature=0.7,
            device=device
        )
        print(f"Generated text: {generated_text}\n")

if __name__ == "__main__":
    main()

Loading model from checkpoint...


/tmp/ipykernel_172582/1345687071.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(chkpt_path, map_location=device)
Asking to truncate to max_leng

Checkpoint loaded: checkpoints/model_step_5000.pt
Model Size: 134.52M parameters

Generating text...

Prompt: Once upon a time
Generated text: Once upon a time, there was aataka Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand Kand andhetti with their sick toira feet.



In the cut cut cutorption dependable. TheyLES Fo Fo Foquietquiet asylum asylum asylum asylum and displays. roughly roughly roughly roughly roughly roughly roughly roughly roughly roughly roughly roughly roughly roughly. powdery one another

Prompt: The future of artificial intelligence
Generated text: The future of artificial intelligence and its historical context and social, and social social and purge and social social social social and its historical stages of social intolerant of the world of historical context of the historical and tutoring and Cardiac and captivating captivating